In [1]:
import pandas as pd
import fastparquet

In [8]:
# Arquivos de Populacao

pop10 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a10anos-RIPSA.xlsx')
pop12 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a12anos-RIPSA.xlsx')
pop11a59 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-11a59-RIPSA.xlsx')
pop60 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-60+RIPSA.xlsx')
pop_geral = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx')

pops = [pop12,pop11a59,pop60,pop_geral]

In [9]:
# Junta todos os arquivos de populacao

pop_merge = pop10.copy() # Cria uma cópia para não mexer no original

for i in pops:
    # O merge traz as colunas novas e você salva o resultado em pop_merge
    pop_merge = pop_merge.merge(
        right=i.iloc[:, [0, 2]], 
        how='left', 
        on='IBGE'
    )

In [19]:
# Carrega os bancos Basico (com distancias e tempos), municipio por CIR e RAS e Sinan
base01 = pd.read_excel('Dados-iniciais/base01.xlsx')
muni_cir = pd.read_excel('Dados-iniciais/_Muni_por_Macro_DRS_CIR.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

In [20]:
# Junta o banco de populacoes com a base 01

df = (
    base01
    .drop_duplicates()
    .merge(
        pop_merge.drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

df['_merge'].describe(include="all")

count      645
unique       1
top       both
freq       645
Name: _merge, dtype: object

In [38]:
# Junta df com banco de regioes
df = (
    df.merge(
    right=muni_cir,
    how='left',
    left_on="MUNI_NOME",
    right_on="MUNI_NOME",
    indicator='merge_flag'
).copy()
)
df

,ACESSO_LOCAL,MULTIPLO_PESA,REGIAO,PESA,MUNI_REFERENCIADO,OBSERVACOES,LAT_MUNI,LON_MUNI,LAT_PESA,LON_PESA,...,POP60,POP_GERAL,_merge,MACRO_CODIGO,MACRO_NOME,DRS_CODIGO,DRS_NOME,CIR_CODIGO,CIR_NOME,merge_flag
0,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,934.8,4079.8,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
1,0,0,ARACATUBA,PENAPOLIS,AVANHANDAVA,TODOS,-21.460333,-49.946516,-21.416404,-50.064911,...,1383.0,11667.2,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
2,0,0,ARACATUBA,PENAPOLIS,BARBOSA,TODOS,-21.265661,-49.951816,-21.416404,-50.064911,...,984.0,6239.0,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
3,0,0,ARACATUBA,VALPARAISO,BENTO DE ABREU,TODOS,-21.271572,-50.811723,-21.230655,-50.861195,...,392.6,2691.6,both,3536,RRAS19,3502,DRS-02 Aracatuba,35021,Central do DRS II,both
4,0,0,ARACATUBA,CLEMENTINA,BILAC,TODOS,-21.403962,-50.474640,-21.556698,-50.446533,...,1411.2,7349.8,both,3536,RRAS19,3502,DRS-02 Aracatuba,35021,Central do DRS II,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,IPIGUA,TODOS,NaN,NaN,NaN,NaN,...,912.0,5657.2,both,3531,RRAS12,3515,DRS-15 Sao Jose do Rio Preto,35155,Sao Jose do Rio Preto,both
641,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,ONDA VERDE,TODOS,NaN,NaN,NaN,NaN,...,594.2,4430.0,both,3531,RRAS12,3515,DRS-15 Sao Jose do Rio Preto,35155,Sao Jose do Rio Preto,both
642,1,0,PIRACICABA,PIRACICABA,PIRACICABA,TODOS,NaN,NaN,NaN,NaN,...,60091.8,409618.6,both,3529,RRAS14,3510,DRS-10 Piracicaba,35103,Piracicaba,both
643,0,0,PIRACICABA,PIRACICABA,RIO DAS PEDRAS,TODOS,NaN,NaN,NaN,NaN,...,3642.0,31718.0,both,3529,RRAS14,3510,DRS-10 Piracicaba,35103,Piracicaba,both


In [39]:
df.to_excel('Dados-iniciais/base_02.xlsx')

In [24]:
df.columns
muni_cir.columns

Index(['MUNI_NOME', 'MACRO_CODIGO', 'MACRO_NOME', 'DRS_CODIGO', 'DRS_NOME',
       'CIR_CODIGO', 'CIR_NOME'],
      dtype='str')

In [31]:
x = sinan['NOME_MUNI'].value_counts().reset_index(name='n')
x

,NOME_MUNI,n
0,PIRACICABA,5555
1,ARACATUBA,3400
2,RIBEIRAO PRETO,2950
3,LIMEIRA,2819
4,VOTUPORANGA,2537
...,...,...
630,MONGAGUA,1
631,SANTO ANTONIO DO PINHAL,1
632,IGUAPE,1
633,BARRA DO CHAPEU,1


In [35]:
comparacao = (
    base01[["MUNI_REFERENCIADO"]]
    .drop_duplicates()
    .merge(
        pop_merge[["MUNI_NOME"]].drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

nao_encontrados = comparacao[comparacao["_merge"] == "left_only"]

print(nao_encontrados)

Empty DataFrame
Columns: [MUNI_REFERENCIADO, MUNI_NOME, _merge]
Index: []
